# SnapList Pro unit-economics validation

Independent, offline checks over the deterministic TypeScript-generated artifact. This notebook performs no provider calls and mutates no account or deployment.

In [1]:
import json
from pathlib import Path

root = Path.cwd().parents[1] if Path.cwd().name == 'unit-economics' else Path.cwd()
model = json.loads((root / 'docs/unit-economics/snaplist-pro-model.json').read_text())
results = json.loads((root / 'docs/unit-economics/snaplist-pro-results.json').read_text())

assert model['status'] == 'provisional-testflight-required'
assert model['boundaries']['productionCommitment'] is False
assert len(model['candidates']) == 3
assert len(results['candidateMatrix']) == 27
direct_costs = [row['directSuccessfulListingCogsUsd'] for row in results['scenarioCosts']]
assert direct_costs == sorted(direct_costs)

balanced_median = [
    row for row in results['candidateMatrix']
    if row['candidateId'] == 'balanced-10' and row['scenarioId'] == 'median'
]
margins = {row['usageCase']: row['monthly']['contributionMarginRate'] for row in balanced_median}
assert margins['low'] > margins['expected'] > margins['high']
assert all(change['contributionMarginDeltaRate'] <= 0 for change in results['sensitivity']['changes'])
assert results['currentGateEvaluation']['status'] == 'not-ready-testflight-evidence-missing'
assert results['currentGateEvaluation']['expectedMedianMonthlyMarginPasses'] is True
assert results['currentGateEvaluation']['p90HighUseMonthlyMarginPasses'] is False
assert results['currentGateEvaluation']['stressMonthlyNonNegative'] is False

print({
    'direct_successful_listing_cogs_usd': direct_costs,
    'balanced_midpoint_monthly_margins': margins,
    'gate_status': results['currentGateEvaluation']['status'],
})

{'direct_successful_listing_cogs_usd': [0.170216, 0.45793, 1.983571], 'balanced_midpoint_monthly_margins': {'low': 0.938636, 'expected': 0.867045, 'high': 0.785227}, 'gate_status': 'not-ready-testflight-evidence-missing'}
